In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option('display.max_columns',None)

In [ ]:
# PART 1: PYTHON(Data cleaning +EDA)

In [ ]:
import os
path= r"C:\Users\user\OneDrive\Desktop\Brazilian_Ecommerce/"

items=os.listdir(path)
print("Contents", items)

if "Brazilian_Ecommerce" in items:
    path=path + "Brazilian_Ecommerce/"
    print("Updated path:", path)

In [ ]:

path= r"C:\Users\user\OneDrive\Desktop\Brazilian_Ecommerce/"

orders= pd.read_csv(path + "olist_orders_dataset.csv")
customers= pd.read_csv(path + "olist_customers_dataset.csv")
items= pd.read_csv(path + "olist_order_items_dataset.csv")
products= pd.read_csv(path + "olist_products_dataset.csv")
payments= pd.read_csv(path + "olist_order_payments_dataset.csv")
reviews= pd.read_csv(path + "olist_order_reviews_dataset.csv")

print(orders.shape, items.shape, customers.shape)

In [ ]:
# Merging
df=orders.merge(items, on="order_id", how="left")
df=df.merge(products, on="product_id", how="left")
df=df.merge(customers, on="customer_id", how="left")
df=df.merge(payments, on="order_id", how="left")

# Convert dates
date_columns=["order_purchase_timestamp","order_delivered_customer_date",
              "order_estimated_delivery_date"]
for col in date_columns:
    df[col]=pd.to_datetime(df[col])

df["delivery_days"]=(df["order_delivered_customer_date"]-df["order_purchase_timestamp"]).dt.days
df["order year"]=df["order_purchase_timestamp"].dt.year
df["order month"]=df["order_purchase_timestamp"].dt.month

print("Master table shape:",df.shape)


In [ ]:
df.columns.tolist()

In [ ]:
print("columns with 'order':")
for col in df.columns:
    if "order" in col.lower():
        print(f" {col} ")

In [ ]:

# Q1 Total Revenue and orders
print("Total Revenue:",df.payment_value.sum())
print("Total Orders:",df.order_id.nunique())

#Q2 Top Products and Categories
print(df.groupby("product_category_name") ["payment_value"].sum().sort_values(ascending= False).head(10))

# Q3 Aveage Delivery Time
print("Avg Delivery Days:", df["delivery_days"].mean())

# Q4 Revenue by state
print(df.groupby("customer_state") ["payment_value"]
      .sum().sort_values(ascending= False).head(10))

# Q5 Monthly sales trend
monthly=df.groupby(["order year", "order month"])["payment_value"].sum()
print(monthly)

In [ ]:
# Dashboard
fig, axes =plt.subplots(2,2, figsize=(14,10))
fig.suptitle("Brazilian E-Commerce - Capstone Dashboard", fontsize=18)

cat=df.groupby("product_category_name")["payment_value"].sum().sort_values().tail(10)
axes[0,0].barh(cat.index,cat.values, color="#2E75B6")
axes[0,0].set_title("Top 10 Categories by Revenue")

state=df.groupby("customer_state")["payment_value"].sum().sort_values(ascending= False).head(10)
axes[0,1].barh(state.index,state.values, color="#2ecc71")
axes[0,1].set_title("Top 10 States by Revenue")

axes[1,0].hist(df["delivery_days"].dropna(), bins=30, color="#e74c3c")
axes[1,0].set_title("Delivery Time Distribution")

monthly_df=df.groupby(["order year", "order month"])["payment_value"].sum().reset_index()
axes[1,1].plot(range(len(monthly_df)), monthly_df["payment_value"],marker="o",color="#9b59b6")
axes[1,1].set_title("Monthly Revenue Trend")

plt.tight_layout()
plt.savefig("Capstone_Dashboard.png", dpi=150)
plt.show()
df.to_csv("ecommerce_cleaned.csv", index=False )






In [ ]:
# Top Categories and its revenue
top_category=df.groupby("product_category_name")["payment_value"].sum().sort_values(ascending= False)
print("Top Category:", top_category.index[0], "-> $", round(top_category.iloc[0],2))

# Top state and its Revenue
top_state=df.groupby("customer_state")["payment_value"].sum().sort_values(ascending= False)
print("Top State:",top_state.index[0], "-> $", round(top_state.iloc[0],2))

# Overall Totals
print("Total Revenue: $", round(df["payment_value"].sum(),2))
print("Total orders:", df["order_id"].nunique())

# Deliver time stats
print("Average Delivery Days:", round(df["delivery_days"].mean(),1))
print("Median Delivery Days:", round(df["delivery_days"].median(),1))

# Monthly Trend
print("\nMonthly revenue(first 3 rows):")
print(monthly_df.head(3))
print("\nMonthly revenue(last 3 rows):")
print(monthly_df.tail(3))

In [ ]:
df.order_purchase_timestamp.max()

In [ ]:
df.to_csv("Brazilian_ecommerce_cleaned" , index=False)
print("Brazilian_ecommerce_cleaned_saved")